# Data Acquisition

**Goal:** Load the PyTorch documentation from `pytorch.org`, clean the HTML, convert to Markdown, and persist to disk with a JSONL metadata index.

## 1. Environment Setup

In [1]:
%load_ext autoreload
%autoreload 2
    
import sys
from pathlib import Path

DATA_DIR = "data"

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", sys.path[0])

from src import data_acquisition as da

Project root: /home/dmitry/Projects/DataScience/rag-techdoc-assistant


In [2]:
import json
import logging
from IPython.display import display, HTML

import re
from urllib.parse import urlparse
from collections import Counter
import markdown as md_lib

# Route all src loggers to the notebook output
logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s  %(name)-20s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

log = logging.getLogger("notebook")

## 2. Configuration

In [3]:
OUTPUT_DIR   = PROJECT_ROOT / DATA_DIR / "pytorch_docs_md"
MAX_PAGES    = None       # set to None to fetch the full documentation
RPS          = 2.0        # requests per second

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INDEX_PATH = OUTPUT_DIR / "_index.jsonl"

print(f"Output dir  : {OUTPUT_DIR}")
print(f"Index file  : {INDEX_PATH}")
print(f"Max pages   : {MAX_PAGES}")
print(f"Rate limit  : {RPS} req/s")

Output dir  : /home/dmitry/Projects/DataScience/rag-techdoc-assistant/data/pytorch_docs_md
Index file  : /home/dmitry/Projects/DataScience/rag-techdoc-assistant/data/pytorch_docs_md/_index.jsonl
Max pages   : None
Rate limit  : 2.0 req/s


## 3. URL Discovery

Fetch the sitemap and materialise the URL list. Inspect the list before committing to a full download run.

In [6]:
urls = da.fetch_url_list(
    sitemap_url=da.SITEMAP_URL,
    skip_patterns=da.DEFAULT_SKIP_PATTERNS,
    max_pages=MAX_PAGES,
)

print(f"\nTotal URLs queued: {len(urls)}")
for u in urls[:5]:
    print(" ", u)
print('\n  ...\n')
for u in urls[-5:]:
    print(" ", u)


Total URLs queued: 2750
  https://docs.pytorch.org/docs/stable/accelerator.html
  https://docs.pytorch.org/docs/stable/accelerator/amp.html
  https://docs.pytorch.org/docs/stable/accelerator/autoload.html
  https://docs.pytorch.org/docs/stable/elastic/train_script.html
  https://docs.pytorch.org/docs/stable/generated/torch.Tensor.arccos.html

  ...

  https://docs.pytorch.org/docs/stable/generated/torch.Tensor.remainder.html
  https://docs.pytorch.org/docs/stable/generated/torch.Tensor.squeeze_.html
  https://docs.pytorch.org/docs/stable/generated/torch.Tensor.remainder_.html
  https://docs.pytorch.org/docs/stable/generated/torch.Tensor.renorm.html
  https://docs.pytorch.org/docs/stable/404.html


Group by inferred section to spot any unexpected content:

In [7]:
def infer_section(url: str) -> str:
    path  = urlparse(url).path
    rel   = re.sub(r"^/docs/stable/", "", path).strip("/")
    first = re.sub(r"\.[^.]+$", "", rel.split("/")[0]) if rel else "root"
    return first or "root"

section_counts = Counter(infer_section(u) for u in urls)
print(f"{'Section':<35} {'Count'}")
print("-" * 43)
for section, count in section_counts.most_common(10):
    print(f"{section:<35} {count:>5}")

Section                             Count
-------------------------------------------
generated                            2506
user_guide                             62
notes                                  25
elastic                                16
accelerator                             8
community                               6
rpc                                     3
fft                                     1
fsdp                                    1
func                                    1


## 4. Single-page Dry Run

Process **one page** end-to-end to inspect the output before running the full pipeline.

In [8]:
SAMPLE_URL = "https://docs.pytorch.org/docs/stable/random.html"
print("Processing:", SAMPLE_URL)

with da.RateLimitedFetcher(requests_per_second=RPS) as fetcher:
    sample_page = da.process_page(SAMPLE_URL, fetcher)

if sample_page:
    print(f"\nTitle      : {sample_page.title}")
    print(f"Section    : {sample_page.section}")
    print(f"Characters : {sample_page.char_count:,}")
    print(f"\n{'─'*60}")
    
    print(sample_page.markdown)
    rendered = md_lib.markdown(
        sample_page.markdown[:3000],
        extensions=["fenced_code", "tables"],
    )
    display(HTML(f"<div style='border:1px solid #ddd; padding:1em'>{rendered}</div>"))
    
else:
    print("[WARNING] Page was skipped (too short or fetch error)")

Processing: https://docs.pytorch.org/docs/stable/random.html

Title      : torch.random
Section    : random
Characters : 2,698

────────────────────────────────────────────────────────────
<!-- section: module-torch.random -->

# torch.random

Created On: Aug 07, 2019 | Last Updated On: Jun 18, 2025

<!-- api: torch.random.fork_rng -->

```python
torch.random.fork_rng(devices=None, enabled=True, _caller='fork_rng', _devices_kw='devices', device_type='cuda')
```
Forks the RNG, so that when you return, the RNG is reset
to the state that it was previously in.

**Parameters**
- **devices** (*iterable* *of* *Device IDs*) – devices for which to fork
  the RNG. CPU RNG state is always forked. By default, `fork_rng()` operates
  on all devices, but will emit a warning if your machine has a lot
  of devices, since this function will run very slowly in that case.
  If you explicitly specify devices, this warning will be suppressed
- **enabled** (*bool*) – if `False`, the RNG is not forked. This 

## 5. Run Entire Pipeline

In [9]:
saved_titles = []

def _on_saved(page: da.DocPage) -> None:
    saved_titles.append(page.title)

pages = da.run_pipeline(
    output_dir=OUTPUT_DIR,
    max_pages=MAX_PAGES,
    requests_per_second=RPS,
    on_page_saved=_on_saved,
    resume=True,
)

print(f"\n✓ Pipeline complete — {len(pages)} pages saved to {OUTPUT_DIR}")

Acquiring docs:   0%|          | 0/2550 [00:00<?, ?page/s]


✓ Pipeline complete — 2750 pages saved to /home/dmitry/Projects/DataScience/rag-techdoc-assistant/data/pytorch_docs_md


## 6. Examine the Output

In [10]:
summary = da.build_summary(pages)

print(f"Total pages : {summary['total_pages']:,}")
print(f"Total chars : {summary['total_chars']:,}")
print(f"Avg chars   : {summary['avg_chars']:,}")
print()
print(f"{'Section':<35} {'Pages':>6}")
print("─" * 43)

for section, count in summary["top_sections"].items():
    print(f"{section:<35} {count:>6}")

Total pages : 2,750
Total chars : 7,215,690
Avg chars   : 2,623

Section                              Pages
───────────────────────────────────────────
generated                             2506
user_guide                              62
notes                                   25
elastic                                 16
accelerator                              8
community                                6
rpc                                      3
fft                                      1
fsdp                                     1
func                                     1


In [11]:
index_records = []
with INDEX_PATH.open(encoding="utf-8") as f:
    for line in f:
        index_records.append(json.loads(line))

print(f"Index records : {len(index_records)}")
print("\nSample record:")
print(json.dumps(index_records[0], indent=2, ensure_ascii=False))

Index records : 2750

Sample record:
{
  "url": "https://docs.pytorch.org/docs/stable/accelerator.html",
  "title": "torch.accelerator",
  "section": "accelerator",
  "markers": {
    "module": "torch.accelerator",
    "symbols": [],
    "sections": [
      {
        "anchor": "#module-torch.accelerator",
        "label": "torch.accelerator",
        "level": 1,
        "kind": "heading",
        "symbol": "",
        "params": [],
        "source_url": ""
      },
      {
        "anchor": "#memory-management",
        "label": "Memory management",
        "level": 2,
        "kind": "heading",
        "symbol": "",
        "params": [],
        "source_url": ""
      }
    ],
    "keywords": [
      "torch.accelerator",
      "torch",
      "accelerator",
      "Memory",
      "management"
    ]
  },
  "char_count": 2541,
  "saved_path": "accelerator.md"
}
